# Security screening: demand scenarios and capacity plan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/security_filters/notebooks/security_filters.ipynb)

How many passengers will arrive at the security checkpoint of an airport, slot by slot, on a summer day of 2008? It depends on assumptions about the flights: how full they are, how many passengers only change planes, and how early passengers come to the airport. In this notebook you write those assumptions as demand scenarios.

**How to use it.** Run the cells in order, with ▶ or *Shift + Enter*. The first cell installs what the notebook needs. Then you work with the controls of each section: you do not need to write code.

On your own computer, open the notebook in Jupyter or PyCharm with the environment of the repository (see the README). In PyCharm, tick *Trust Notebook* so that the controls work.

In [ ]:
# @title Setup (run this cell first) { display-mode: "form" }
# In Google Colab, install the package from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/security_filters"
    %pip install -q {url}
    # A session that already had the package keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    if "security_filters" in sys.modules:
        print("The package was already loaded in this session: restart it "
              "(Runtime > Restart session) and run this cell again.")
from security_filters.panel import ComparisonPanel, PlanPanel, PPPPanel, QueuePanel, ScenarioPanel

## 1. Demand scenarios

The next cell opens a panel to build a demand scenario: the assumptions on the flights of one day.

1. **Day.** Choose the day you study. Its flights are loaded, and also the flights of the next day that leave before 02:55, marked *+1*, because some of their passengers arrive before midnight.
2. **Select a group of flights** with the lists *Airlines* and *Destinations*. Ctrl + click (Cmd + click on a Mac) selects several items or unselects one. Choosing airlines narrows the list of destinations to those they fly to, and the other way round. With nothing selected, all the flights are selected. The table shows the selected flights and their values.
3. **Give values to the group.** Tick the values you want to change, set them and press **Assign**. Only the ticked values change.
   - *Load factor*: share of the seats that are occupied, from 0 to 1.
   - *Transit share*: share of the passengers who only change planes and do not go through security screening, from 0 to 1.
   - *Arrival pattern*: how long before departure the passengers arrive, as a probability function of time in 5-minute slots. Choose one of the five patterns of the list, or a normal or an Erlang function with your own parameters.

   Every flight starts with load factor 1, transit share 0, the pattern *Domestic, mixed purpose* and no arrival surge. If two groups have flights in common, those flights keep the values of the last **Assign**.
4. **Arrival surges.** Passengers of several flights of one airline who arrive at the airport together, for example on the buses of a tour operator. Select the airline, tick the flights of the surge in the list *Flights* and press **Make arrival surge**. The flight that departs first is the reference. The *offset* of each flight is the number of 5-minute slots between its departure and the departure of the reference. An airline can have several surges at different times of the day, but a flight is only in one.
5. **Scenario.** Every **Assign**, surge and removal is a step of the scenario, listed in order. Write a name and press **Save scenario**. **Load** opens a saved scenario again. **Download** saves it as a file on your computer, and **Upload** reads it back: Colab deletes its files when the session ends, so download the scenarios you want to keep.

In [ ]:
panel = ScenarioPanel()
panel

## 2. PPP of the airport

The next cell computes the PPP of the airport: the passengers who arrive at the security checkpoint in each 5-minute slot of the day, with the values that the flights have in a scenario.

1. **Choose the scenario**: the one you are building in section 1 (*Current scenario*) or one you saved. If you saved a scenario after running this cell, press **Refresh list**. Then press **Compute PPP**.
2. **Read the result.** The chart shows the passengers in each 5-minute slot, from 00:00 to 24:00. Below it you find the passengers of the day, the passengers who arrive outside the day, the highest value with its slot, the busiest hour, and a table with the passengers of each hour. The passengers outside the day belong to the flights of early morning: some passengers of the first flights of the day arrive before 00:00, and some of the flights of the next day arrive after 24:00.
3. **See where the PPP comes from.** Tick **Show the selected flights**: the chart adds, in orange, the PPP of the flights selected in section 1 (its airlines and destinations, or the flights ticked in its list *Flights*), and the text says how many of the passengers of the highest value of the airport they bring. When you change the selection in section 1, the chart changes too.
4. **Download the 288 slots** saves the PPP as a file (`ppp.csv`).

How it is computed. A flight with $A$ seats, load factor $L$ and transit share $T$ brings $Q = B - R$ passengers to the security checkpoint, with $B = \operatorname{round}(L A)$ passengers on board and $R = \operatorname{round}(T B)$ in transit. Its arrival pattern spreads them over the 29 slots before the gate closes, which is 30 minutes before departure, in whole passengers. An arrival surge moves the pattern earlier by the offset of the flight.

In [ ]:
ppp_panel = PPPPanel(panel)
ppp_panel

## 3. Comparison of scenarios

The next cell shows the PPPs of several scenarios of the same day together, so that you can analyse them.

1. **Choose the scenarios.** The list has the scenario you are building in section 1 (*Current scenario*) and the scenarios you saved, also those you uploaded from a file. If you saved a scenario after running this cell, press **Refresh list**. Select from two to six scenarios of the same day (Ctrl + click, or Cmd + click on a Mac) and press **Compare**.
2. **Read the chart.** Each scenario has its own colour and type of line, with its name in the legend. **From** and **To** show only a part of the day.
3. **Read the tables.** The first table gives, for each scenario, the same summary as section 2: passengers in the day, passengers who arrive outside the day, highest value and its slot, and busiest hour. The second one gives the passengers of each hour. The scenarios are in the order of the list.
4. **Add the real PPP.** After the day, tick **Add the real PPP** to add the real PPP of the day to the chart (black line) and to the tables (column *Real PPP*). Its summary has no passengers outside the day, because only the passengers of the day are known.
5. **Download the 288 slots** saves the PPPs as a file (`ppp_comparison.csv`), with one column per scenario.

In [ ]:
comparison_panel = ComparisonPanel(panel)
comparison_panel

## 4. Capacity plan

The next cell lets you write a capacity plan for security screening and draw it over the PPPs of your scenarios. The airport sets the capacity before the day starts, so the plan can only rely on the scenarios.

1. **Parameters.** *Lanes available* ($L$) is the number of screening lanes, and *Capacity of a lane* ($C_L$) the passengers that one lane serves in a 5-minute slot. The maximum capacity, with every lane open, is $C_{max} = L \times C_L$. You can change both.
2. **Periods.** The plan divides the day into periods that start on the hour. Each period keeps from 1 to $L$ lanes open, and the capacity of each of its 5-minute slots is the open lanes times $C_L$. The first period starts at 00:00, and the last one ends at 24:00. **Add period** adds a period, and ✕ removes it. Each row shows the capacity of its slots and its share of $C_{max}$.
3. **Draw the plan.** Select up to six scenarios of the same day in the list (or none) and press **Draw plan**. The chart shows the capacity of the plan as a thick line, the PPPs of the scenarios as thin lines, and $C_{max}$ as a dashed line. The table below gives each period with its open lanes and its capacity per slot, and the message gives the capacity of the whole day. If you saved a scenario after running this cell, press **Refresh list**.
4. **Plans.** Write a name and press **Save plan**. **Load** opens a saved plan again. **Download** saves it as a file (`plan_<name>.json`), with $L$, $C_L$ and its periods, and **Upload** reads it back.

In [ ]:
plan_panel = PlanPanel(panel)
plan_panel

## 5. Check against the real PPP

After the day, the real PPP is known. The next cell applies a capacity plan to it and shows the queue at security screening, the unused capacity and the waits.

1. **Choose the plan, the demand and the day.** The list *Plan* has the plan you are writing in section 4 (*Current plan*) and the plans you saved or uploaded; if you saved a plan or a scenario after running this cell, press **Refresh list**. The list *Demand* has the real PPP and the scenarios you saved in section 1; with a scenario, the day is that of the scenario. Choose the day and press **Apply plan**.
2. **How the queue works.** In each 5-minute slot, the passengers who arrive join the queue, and the open lanes let through as many passengers as the capacity of the slot. Those who cannot go through wait for the next slot, in order of arrival. The queue starts empty at 00:00. The capacity that is left when there is nobody waiting is the unused capacity. The wait of a passenger is the number of slots between the slot in which he or she arrives and the slot in which he or she goes through, times 5 minutes (0 if both are the same slot). If there is still a queue at 24:00, it goes through after midnight with the capacity of the last period, and those waits count too.
3. **Read the result.** The upper chart shows the demand and the capacity of the plan; the lower one shows the queue at the end of each slot and the unused capacity. Below them: the passengers of the day, the mean wait, the longest wait (and when those passengers arrived), the highest queue and its slot, the unused capacity of the day, and the queue at 24:00. The table gives the arrivals, the capacity, the passengers who go through, the unused capacity and the queue at the end of each hour.
4. **Download the 288 slots** saves the slots of the day as a file (`queues.csv`).
5. **Passengers who miss the closing of their gate.** Choose the demand to which the plan is applied: the real PPP of the day or one of your saved scenarios. Below the summary, the tool counts the passengers who would go through security screening after the gate of their flight has closed. You can set the time of the walk from the security checkpoint to the gate (0 minutes by default). Passengers who reach the checkpoint less than that time before the gate closes would miss it even with no queue, so they are counted apart. The passengers who arrive in the same slot go through security screening in a mixed order, and nobody leaves the queue. With a scenario, the tool lists the flights affected. With the real PPP it gives the passengers by hour of departure only, because the real PPP comes without its flights. All the figures are those of the simulated airport of the exercise.

In [ ]:
queue_panel = QueuePanel(panel, plan_panel)
queue_panel